# Reproducing Holdgraf et al. (2016) from CRCNS ac-4

Paper: Holdgraf, de Heer, Pasley, Rieger, Crone, Lin, Knight & Theunissen (2016), "Rapid tuning
shifts in human auditory cortex enhance speech intelligibility", Nat Commun 7:13654,
doi:10.1038/ncomms13654.

Dataset: CRCNS ac-4 (doi:10.6080/K08S4N4J), "Electrocorticography recordings from patients during
a passive listening task of degraded and intact English speech", read in place from
`ac-4/live/data.zip` in the public S3 bucket `crcnsarchive`.

Version: CRCNS data sets are not versioned, so the version is recorded as `unversioned`. The
bucket holds one copy of the data set (`ac-4/live/`, no `old/` folder), the data descriptor's
version 0.6 of 28 May 2019, which is the only published version and the one the paper's
data availability statement points to. The MD5 of `data.zip` in the data set's `checksums.md5`
is printed below, so a later change is detectable.

Claims: `claims/crcns-ac-4/10.1038-ncomms13654.yaml`, written from the version of record with its
Supplementary Information. The claims file decides what is tested; method details it lacks are
taken from the paper and cited where they are used (in this notebook or in `ac4.py`).

| Claim | Tested here |
|---|---|
| behav-no-context-baseline, behav-context-enhancement, supp1a-linguistic-context, supp1a-speaker-no-difference, supp1b-filter-type, supp5a-behaviour-over-trials | Blocked: the behavioural control data are not in the data set |
| speech-r-count | Tested |
| fig3b-hfb-increase-after | Tested |
| supp3a-pink-noise-no-hfb-change | Tested |
| supp5b-hfb-stable-over-session | Tested |
| fig4a-coherence-after-higher, fig4b-coherence-sentence-specific | Tested |
| strf-r-count, included-electrode-count | Tested |
| fig8-mtf-matches-filter | Tested |
| fig9-rms-increase, fig9-middle-generalizes-after | Tested |
| fig10-partial-correlation, fig10b-similarity-correlates, fig10c-stg-location | Tested |
| supp3b-pink-noise-no-pcorr-change | Run, but no electrode of the pink noise subjects is included, so it is blocked |
| supp6a-connectivity-increase, supp6a-after-closer-to-middle, supp6b-no-pac-change | Tested |
| supp7-envelope-coherence | Tested |
| supp4-hfb-power-increase | Tested |

The whole notebook streams about 1.6 GB and takes about 13 minutes, most of it loading.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time
import resource
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy import stats
import ac4

mne.set_log_level("ERROR")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print(ac4.DATASET, "doi:" + ac4.DOI)
print(urllib.request.urlopen(ac4.BASE + "/checksums.md5").read().decode())

In [ ]:
# LOAD: stream each subject, reduce it to HFB at 50 Hz, the auditory spectrograms and speech
# envelopes of every sound, and the re-referenced signal around each sound (see ac4.load_subject)
t0 = time.time()
z = ac4.open_zip()
D = {}
for s in ac4.SUBJECTS:
    D[s] = ac4.load_subject(z, s)
    print(s, f"{time.time() - t0:.0f} s", f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check

In [ ]:
def region(desc):
    if desc.startswith("temporal") or desc == "sub_temporal":
        return "temporal"
    if desc.endswith("-inferior"):
        return "perisylvian"          # inferior frontal, premotor, motor, parietal
    if desc.startswith("frontal") or desc.startswith("premotor"):
        return "frontal"
    return "other"

rows = []
for s, d in D.items():
    ch, tr, js = d["channels"], d["trials"], d["json"]
    rows.append(dict(subject=s, channels=len(ch), good=int((ch.status == "good").sum()),
                     fs=js["SamplingFrequency"], duration_json=js["RecordingDuration"],
                     duration_samples=d["stats"]["n_samples"] / ac4.FS,
                     audio_sr=d["audio_sr"], audio_s=d["audio_len"] / d["audio_sr"],
                     triplets=tr.loc[tr.trial_num >= 0, "trial_num"].nunique(),
                     noise_triplets=tr.loc[tr.noise, "trial_num"].nunique(),
                     lone_unfiltered=int((tr.trial_num < 0).sum()),
                     sound_s=f"{tr.duration.min():.2f}-{tr.duration.max():.2f}",
                     bad_spans=int((d["events"].trial_type == "bad").sum()),
                     sounds_overlapping_bad=int(tr.overlaps_bad.sum()),
                     raw_abs_median=d["stats"]["raw_abs_median"],
                     car_residual=d["stats"]["car_residual"]))
summary = pd.DataFrame(rows)
print(summary.to_string())
print("total channels", summary.channels.sum(), "good", summary.good.sum(), "(paper: 468 kept of 524)")
print("units in channels.tsv:", {s: d["channels"].units.unique().tolist() for s, d in D.items()})
print("reference in channels.tsv:", {s: d["channels"].reference.unique().tolist() for s, d in D.items()})

E = pd.concat([d["good"] for d in D.values()], ignore_index=True)
E["region"] = E.description.map(region)
E["stg"] = E.description.eq("temporal-superior")
print(pd.crosstab(E.subject, E.region, margins=True))
print(E.description.value_counts().to_string())

cond = pd.concat([d["trials"].assign(subject=s) for s, d in D.items()])
print(pd.crosstab([cond.subject, cond.noise], [cond.trial_phase, cond.filt_type]))
# every triplet has one sound of each phase, the same filter BEFORE and AFTER, and similar durations
for s, d in D.items():
    t = d["trials"][d["trials"].trial_num >= 0]
    g = t.groupby("trial_num")
    ok_phase = (g.trial_phase.apply(lambda x: sorted(x) == ["aft", "bef", "mid"])).sum()
    filt_same = g.apply(lambda x: x.loc[x.trial_phase == "bef", "filt_type"].tolist() == x.loc[x.trial_phase == "aft", "filt_type"].tolist()).sum()
    order = g.apply(lambda x: list(x.sort_values("onset").trial_phase) == ["bef", "mid", "aft"]).sum()
    print(s, "triplets", g.ngroups, "complete", ok_phase, "same filter BEFORE/AFTER", filt_same, "in order BEFORE-MIDDLE-AFTER", order)

# HFB range
for s, d in D.items():
    h = d["hfb"]
    print(s, "HFB shape", h.shape, "at", ac4.FS_OUT, "Hz;", f"range {h.min():.3g} to {h.max():.3g} (stored units)")

The checks above, stated:

- Sampling rate 1000 Hz in every `ieeg.json`, and sample counts give the stated recording
  durations (columns `duration_json` and `duration_samples`).
- The stored values are not in the stated millivolts: the median absolute value is about 300 in
  S01 and 1.7e8 to 6.2e8 in the other subjects. The analysis uses only z-scored or normalized
  measures, so the scale does not enter any result; it is recorded as a finding.
- The good channels are already re-referenced to their common average (the mean over good
  channels is about 1e-8 of the signal), as `channels.tsv` says (`CAR`).
- The paper reports 7 subjects with grids of 48, 64, 68, 68, 62, 110 and 104 channels and 468
  electrodes kept; the data set has 7 subjects with 416 channels, 377 marked good.
- Three subjects (S07, S08, S13) have pink noise triplets, about half of their triplets, as the
  paper says.

Clock checks follow: the audio track against the event onsets, and the HFB against the onsets,
with the onsets shifted as the contrast.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for s, d in D.items():
    t, a = d["audio_onset_profile"]
    pre, post = a[(t < -0.05)].mean(), a[(t > 0.05) & (t < 0.5)].mean()
    ax[0].plot(t, a / post, lw=0.8, label=s)
    print(s, f"mean |audio| before onsets {pre:.2e}, 0.05-0.5 s after {post:.2e}, ratio {post / pre:.1f}")
ax[0].set(xlabel="time from sound onset (s)", ylabel="mean |audio| (relative)", title="audio at event onsets")
ax[0].legend(fontsize=6)

# HFB clock check: on temporal electrodes, mean HFB 0.1-0.5 s after onset minus -0.4-0 s,
# at the true onsets and at onsets shifted by -2 to +2 s
shifts = np.arange(-2.0, 2.01, 0.1)
for s, d in D.items():
    temp = (d["good"].description.map(region) == "temporal").to_numpy()
    h = d["hfb"][temp]
    on = d["trials"].onset.to_numpy()
    con = []
    for sh in shifts:
        i = np.round((on + sh) * ac4.FS_OUT).astype(int)
        post = np.mean([h[:, j + 5:j + 25].mean() for j in i])
        pre = np.mean([h[:, j - 20:j].mean() for j in i])
        con.append((post - pre) / h.std())
    con = np.array(con)
    ax[1].plot(shifts, con, lw=0.8)
    print(s, f"HFB contrast at true onsets {con[np.argmin(np.abs(shifts))]:.3f}; largest at shift {shifts[np.argmax(con)]:+.1f} s; range {con.min():.3f} to {con.max():.3f}")
ax[1].set(xlabel="onset shift (s)", ylabel="HFB post minus pre (s.d.)", title="HFB at shifted onsets")
plt.tight_layout()

## Electrode selection (Speech-R)

Methods, Evoked HFB and speech-responsive electrodes: each trial baselined at -800 to -100 ms,
the mean post-stimulus activity per trial, a bootstrap 99% interval of the mean over trials, and
Speech-R when its 0.5th percentile is above 0 in response to unfiltered speech. The claims file
names three open details (1 trials, 2 window, 3 z-score); every combination is run.

Triplets with pink noise are left out of every speech analysis (Figure 3 heading "MIDDLE
(context) unfiltered speech only"). The six unfiltered sentences outside any triplet in S01 to
S04 (trial number -1) are left out too: they are not MIDDLE sounds of the task.

> speech-r-count: 92 of the 468 electrodes (19.6%) responded to speech above baseline (Speech-R),
> generally centred on perisylvian regions.

In [ ]:
rng = np.random.default_rng(101)
EP = {}
for s, d in D.items():
    EP[s] = ac4.hfb_epochs(d)
POST = {}
readings = []
for zmode in ["trial", "pooled", "nonevent"]:
    for win in ["sound", "2.5"]:
        for tsel in ["mid", "all"]:
            lbs = []
            for s, d in D.items():
                ep, times = EP[s]
                tr = d["trials"]
                zep = ac4.zscore_epochs(d, ep, times, zmode)
                pm = ac4.post_mean(zep, times, tr.duration.to_numpy(), win)
                POST[(s, zmode, win)] = pm
                sel = (tr.trial_num >= 0) & ~tr.noise
                if tsel == "mid":
                    sel &= tr.trial_phase == "mid"
                lbs.append(ac4.bootstrap_lower(pm[sel.to_numpy()], rng))
            lb = np.concatenate(lbs)
            E[f"lb_{zmode}_{win}_{tsel}"] = lb
            n = int((lb > 0).sum())
            readings.append(dict(zscore=zmode, window=win, trials=tsel, speech_r=n,
                                 pct_of_good=100 * n / len(E), within_92_tol=abs(n - 92) <= 9.2,
                                 **{s: int((lb[E.subject.eq(s).to_numpy()] > 0).sum()) for s in D}))
R_speech = pd.DataFrame(readings)
print(R_speech.to_string())
print(f"electrodes: {len(E)} good of {summary.channels.sum()} in the data set; paper 468 (tolerance 421.2-514.8)")
print("readings within 92 +/- 9.2:", int(R_speech.within_92_tol.sum()), "of", len(R_speech))

MAIN = ("nonevent", "2.5", "mid")
E["speech_r"] = E[f"lb_{MAIN[0]}_{MAIN[1]}_{MAIN[2]}"] > 0
print("adopted reading", MAIN, "Speech-R", int(E.speech_r.sum()))
print(pd.crosstab(E.region, E.speech_r))

The reading carried into the later claims is the z-score by non-event times (the code's
`zscore_by_non_events`), the 0 to 2.5 s window, and MIDDLE trials (the Methods): of the readings
within the paper's count, it is the first in the order the claims file lists them. Only the
non-event z-score gives counts near the paper's; see the record's assumptions. Sounds that
overlap a span marked `bad` in the events table are kept, as the paper does not mention such
spans (column `sounds_overlapping_bad` above).

**Result: speech-r-count.** Three of the twelve readings give a count within 92 +/- 9.2, all with the z-score by non-event times: 96 (onset to sound end, all speech trials), 101 (0 to 2.5 s, MIDDLE trials) and 99 (0 to 2.5 s, all trials). The z-score by each trial's own baseline or by the pooled baseline gives 131 to 185. The data set holds 377 good electrodes (416 in all), against the paper's 468, so 101 is 26.8% of the electrodes against the paper's 19.6%. Speech-R electrodes lie mostly on temporal (66) and perisylvian (18) sites. The count is reproduced under the non-event z-score; the electrode total differs.

## HFB increase AFTER against BEFORE

> fig3b-hfb-increase-after: Across Speech-R electrodes, HFB activity is higher in the AFTER
> condition than in the BEFORE condition (cluster-based permutation test, P = 0.003, n = 92).

Per electrode, the trial-averaged AFTER minus BEFORE time course of z-scored HFB (Methods, Evoked
HFB: "the difference in time-varying HFB activity in each trial was calculated and then averaged
across trials"), from -0.5 to 3 s (the Figure 3b axis), and MNE's one-sample cluster permutation
test across electrodes (claims, open detail 11: default threshold, 10,000 permutations).

In [ ]:
def diff_timecourse(zmode, sel_fn, t0=-0.5, t1=3.0):
    """(electrodes, time) of mean AFTER minus mean BEFORE z-HFB, for the electrodes of E in order."""
    out = []
    for s, d in D.items():
        ep, times = EP[s]
        zep = ac4.zscore_epochs(d, ep, times, zmode)
        tr = d["trials"]
        keep = (times >= t0) & (times <= t1)
        b = sel_fn(tr) & (tr.trial_phase == "bef")
        a = sel_fn(tr) & (tr.trial_phase == "aft")
        out.append(zep[a.to_numpy()][..., keep].mean(0) - zep[b.to_numpy()][..., keep].mean(0))
    return np.concatenate(out), times[keep]

def as_mask(cl, shape):
    m = np.zeros(shape, bool)
    m[cl] = True
    return m

def clusters_of(X, seed, n_perm=10000):
    """MNE's one-sample cluster permutation test (default threshold, two-tailed); masks."""
    T, clusters, pv, _ = mne.stats.permutation_cluster_1samp_test(X, n_permutations=n_perm, tail=0, seed=seed, out_type="mask")
    return T, [as_mask(c, X.shape[1:]) for c in clusters], pv

def cluster_report(X, label, seed, times=None, n_perm=10000):
    T, clusters, pv = clusters_of(X, seed, n_perm)
    rows = []
    for c, p in zip(clusters, pv):
        idx = np.where(c)[0] if c.ndim == 1 else None
        rows.append(dict(set=label, n=X.shape[0], p=p, sign="+" if T[c].mean() > 0 else "-",
                         start=None if idx is None or times is None else times[idx[0]],
                         end=None if idx is None or times is None else times[idx[-1]],
                         size=int(c.sum())))
    return pd.DataFrame(rows) if rows else pd.DataFrame([dict(set=label, n=X.shape[0], p=np.nan, sign="none", start=None, end=None, size=0)])

speech = lambda tr: (tr.trial_num >= 0) & ~tr.noise
out = []
for zmode in ["nonevent", "trial"]:
    DIF, tt = diff_timecourse(zmode, speech)
    for label, m in [("(a) all Speech-R", E.speech_r), ("(b) Speech-R temporal+perisylvian", E.speech_r & E.region.isin(["temporal", "perisylvian"]))]:
        r = cluster_report(DIF[m.to_numpy()], label, 3, tt)
        r["zscore"] = zmode
        out.append(r)
        if zmode == "nonevent" and label.startswith("(a)"):
            DIF_main = DIF[m.to_numpy()]
R3b = pd.concat(out, ignore_index=True)
print(R3b.to_string())
mean_d = DIF_main.mean(0)
print(f"mean AFTER-BEFORE over electrodes (a): range {mean_d.min():.3f} to {mean_d.max():.3f} z; 0-1 s mean {mean_d[(tt>=0)&(tt<1)].mean():.3f}, 1-3 s mean {mean_d[tt>=1].mean():.3f}")
for s in D:
    m = (E.subject.eq(s) & E.speech_r).to_numpy()
    sub = DIF_main[(E.subject[E.speech_r] == s).to_numpy()]
    print(s, "Speech-R", int(m.sum()), "mean AFTER-BEFORE 0-3 s", f"{sub[:, tt >= 0].mean():.3f}" if len(sub) else "n/a")
fig, ax = plt.subplots(figsize=(6, 3))
se = DIF_main.std(0) / np.sqrt(len(DIF_main))
ax.plot(tt, mean_d, color="purple")
ax.fill_between(tt, mean_d - se, mean_d + se, color="purple", alpha=0.3)
ax.axhline(0, color="k", lw=0.5)
ax.set(xlabel="time (s)", ylabel="AFTER - BEFORE HFB (z)", title=f"Speech-R electrodes, n = {len(DIF_main)}")

**Result: fig3b-hfb-increase-after.** Over the 101 Speech-R electrodes, AFTER is higher than BEFORE in a cluster from 0.36 to 1.28 s (P = 0.0001) and in a second from 1.72 to 2.02 s (P = 0.0072); the 84 Speech-R electrodes on temporal or perisylvian cortex give a cluster from 0.40 to 1.42 s (P = 0.0001), and the per-trial baseline z-score gives the same (P = 0.0002). The mean difference is 0.084 z over 0 to 1 s. Two of seven subjects (S02, S13) have a slightly negative mean difference. Reproduced.

## Pink noise triplets: no HFB change

> supp3a-pink-noise-no-hfb-change: In the three subjects with pink noise trials, evoked HFB does
> not differ between AFTER and BEFORE when the MIDDLE sound was pink noise.

The same test on the pink noise triplets of S07, S08 and S13. The electrode set (a), the
electrodes included in the eSTRF analyses, is computed below with the eSTRF selection; it is
reported in that section. Set (b), the Speech-R electrodes of the three subjects, is here.

In [ ]:
noise_tr = lambda tr: (tr.trial_num >= 0) & tr.noise
noise_subjects = ["S07", "S08", "S13"]
def noise_diff(zmode="nonevent"):
    out = []
    for s in noise_subjects:
        d = D[s]
        ep, times = EP[s]
        zep = ac4.zscore_epochs(d, ep, times, zmode)
        tr = d["trials"]
        keep = (times >= -0.5) & (times <= 3.0)
        b = (noise_tr(tr) & (tr.trial_phase == "bef")).to_numpy()
        a = (noise_tr(tr) & (tr.trial_phase == "aft")).to_numpy()
        out.append(zep[a][..., keep].mean(0) - zep[b][..., keep].mean(0))
    return np.concatenate(out), times[keep]
DIFN, ttn = noise_diff()
EN = E[E.subject.isin(noise_subjects)].reset_index(drop=True)
r = cluster_report(DIFN[EN.speech_r.to_numpy()], "(b) Speech-R of S07, S08, S13", 5, ttn)
print(r.to_string())
print("speech triplets of the same subjects and electrodes, for comparison:")
DIFS = diff_timecourse("nonevent", speech)[0][E.subject.isin(noise_subjects).to_numpy()]
print(cluster_report(DIFS[EN.speech_r.to_numpy()], "(b) speech triplets", 5, ttn).to_string())
print(f"pink noise AFTER-BEFORE mean over (b), 0-3 s: {DIFN[EN.speech_r.to_numpy()][:, ttn >= 0].mean():.3f} z; speech: {DIFS[EN.speech_r.to_numpy()][:, ttn >= 0].mean():.3f} z")

## HFB over the session

> supp5b-hfb-stable-over-session: Mean HFB activity of Speech-R electrodes shows no significant
> relation to trial number over a session.

The paper's test is a bootstrap of the regression slope on trial number (ecogtools
`bootstrap_coefficients`, 1000 resamples, as the claims file gives it). The value is the mean
post-stimulus HFB of each speech sound (all three phases, the adopted z-score and window), and
trial number the triplet number. Two units, as the criterion names: (a) every Speech-R
electrode's value per sound, pooled; (b) the mean over Speech-R electrodes per sound and subject.

In [ ]:
rng = np.random.default_rng(505)
def boot_slope(x, y, rng, n=1000):
    x, y = np.asarray(x, float), np.asarray(y, float)
    slopes = np.empty(n)
    for i in range(n):
        j = rng.integers(0, len(x), len(x))
        slopes[i] = np.polyfit(x[j], y[j], 1)[0]
    return np.polyfit(x, y, 1)[0], np.percentile(slopes, [2.5, 97.5])
xa, ya, xb, yb, rows = [], [], [], [], []
for s, d in D.items():
    tr = d["trials"]
    sel = speech(tr).to_numpy()
    pm = POST[(s, MAIN[0], MAIN[1])][sel][:, E.loc[E.subject.eq(s), "speech_r"].to_numpy()]
    tn = tr.trial_num.to_numpy()[sel]
    if pm.shape[1] == 0:
        continue
    xa.append(np.repeat(tn, pm.shape[1])); ya.append(pm.ravel())
    xb.append(tn); yb.append(pm.mean(1))
    sl, ci = boot_slope(tn, pm.mean(1), rng)
    rows.append(dict(subject=s, sounds=len(tn), speech_r=pm.shape[1], slope=sl, lo=ci[0], hi=ci[1]))
for lab, x, y in [("(a) electrode x sound", xa, ya), ("(b) mean over electrodes per sound", xb, yb)]:
    sl, ci = boot_slope(np.concatenate(x), np.concatenate(y), rng)
    print(f"{lab}: slope {sl:.4f} z per trial, 95% bootstrap interval {ci[0]:.4f} to {ci[1]:.4f}, includes 0: {ci[0] <= 0 <= ci[1]}")
print("per subject (mean over electrodes):")
print(pd.DataFrame(rows).to_string())

**Result: supp5b-hfb-stable-over-session.** The slope of mean HFB on trial number is negative and its 95% bootstrap interval excludes 0 under both units: -0.0022 z per trial (-0.0028 to -0.0017) for electrode values pooled, and -0.0019 (-0.0029 to -0.0009) for the mean over electrodes. Six of seven subjects have an interval below 0. Over a session of 57 triplets this is a decrease of about 0.1 z. The paper's test finds the effect here, so the claim of absence is not reproduced.

## Between-condition coherence

> fig4a-coherence-after-higher / fig4b-coherence-sentence-specific

Methods, Between-condition coherence: per sentence, the coherence of the HFB time course between
BEFORE and MIDDLE and between AFTER and MIDDLE, in 400 ms windows stepped by 200 ms from -500 to
2500 ms, jackknife-corrected, converted to bits/s, averaged over trials; then the sign-flip test
of AFTER/MIDDLE minus BEFORE/MIDDLE across electrodes for the windows of Figure 4 (-1 to 0, 0 to
1, 1 to 2.5 s; a window is assigned by its centre). The multitaper bandwidth is open (claims,
open detail 10): a 400 ms window at 50 Hz holds 20 samples, and MNE's default bandwidth (4 Hz)
gives a single taper, for which single-trial coherence is 1. The choice here is NW = 2 (three
tapers, 10 Hz bandwidth), jackknife over tapers, and the integral over 2.5 to 25 Hz. Electrode
sets: (a) Speech-R on temporal or perisylvian cortex, (b) all temporal or perisylvian electrodes.
Panel b subtracts from each trial the electrode's mean time course, (a) over all trials of all
three conditions, (b) per condition.

In [ ]:
def complete_triplets(tr):
    """Speech triplets with one BEFORE, one MIDDLE and one AFTER sound."""
    t = tr[speech(tr)]
    g = t.groupby("trial_num").trial_phase.apply(lambda x: sorted(x) == ["aft", "bef", "mid"])
    return sorted(g.index[g])

WIN_STARTS = np.round(np.arange(-0.5, 2.1001, 0.2), 1)
WIN_CENTRES = WIN_STARTS + 0.2
def coherence_bits(subtract=None):
    """(electrodes of E, 2 pairs, windows) mean over triplets of integrated coherence in bits/s."""
    out = []
    for s, d in D.items():
        ep, times = EP[s]
        zep = ac4.zscore_epochs(d, ep, times, MAIN[0])
        tr = d["trials"]
        tsel = speech(tr)
        trip = complete_triplets(tr)
        ix = {ph: [tr.index[(tr.trial_num == n) & (tr.trial_phase == ph)][0] for n in trip] for ph in ["bef", "mid", "aft"]}
        X = {ph: zep[ix[ph]] for ph in ix}
        if subtract == "all":
            m = zep[tsel.to_numpy()].mean(0)
            X = {ph: X[ph] - m for ph in X}
        elif subtract == "condition":
            X = {ph: X[ph] - X[ph].mean(0) for ph in X}
        res = np.empty((zep.shape[1], 2, len(WIN_STARTS)))
        for w, ws in enumerate(WIN_STARTS):
            sel = (times >= ws - 1e-9) & (times < ws + 0.4 - 1e-9)
            for k, ph in enumerate(["bef", "aft"]):
                f, c = ac4.mt_coherence(X[ph][..., sel], X["mid"][..., sel], ac4.FS_OUT, nw=2.0)
                res[:, k, w] = ac4.coh_to_bits(c, f).mean(0)
        out.append(res)
    return np.concatenate(out), f

def window_tests(C, mask, rng, label):
    rows = []
    for name, lo, hi in [("-1 to 0 s", -1, 0), ("0 to 1 s", 0, 1), ("1 to 2.5 s", 1, 2.6)]:
        w = (WIN_CENTRES >= lo) & (WIN_CENTRES < hi)
        diff = (C[mask][:, 1, w] - C[mask][:, 0, w]).mean(1)
        obs, p = ac4.signflip_test(diff, rng)
        rows.append(dict(set=label, window=name, n=int(mask.sum()), windows=int(w.sum()),
                         bef_mid=C[mask][:, 0, w].mean(), aft_mid=C[mask][:, 1, w].mean(),
                         diff=obs, se=diff.std(ddof=1) / np.sqrt(len(diff)), p=p))
    return rows

rng = np.random.default_rng(404)
C4a, fr = coherence_bits(None)
print("coherence frequencies (Hz):", fr)
peri = E.region.isin(["temporal", "perisylvian"]).to_numpy()
sets4 = {"(a) Speech-R temporal+perisylvian": peri & E.speech_r.to_numpy(), "(b) all temporal+perisylvian": peri}
R4a = pd.DataFrame(sum([window_tests(C4a, m, rng, k) for k, m in sets4.items()], []))
print(R4a.to_string())
print("per subject, set (a), 0-2.5 s AFTER/MIDDLE minus BEFORE/MIDDLE (bits/s):")
for s in D:
    m = sets4["(a) Speech-R temporal+perisylvian"] & E.subject.eq(s).to_numpy()
    w = WIN_CENTRES >= 0
    print(" ", s, int(m.sum()), f"{(C4a[m][:, 1, w] - C4a[m][:, 0, w]).mean():.3f}" if m.any() else "n/a")
m = sets4["(a) Speech-R temporal+perisylvian"]
fig, ax = plt.subplots(figsize=(6, 3))
for k, lab in enumerate(["BEFORE/MIDDLE", "AFTER/MIDDLE"]):
    ax.plot(WIN_CENTRES, C4a[m][:, k].mean(0), label=lab)
ax.set(xlabel="window centre (s)", ylabel="coherence (bits/s)")
ax.legend()
print(f"plotted range {C4a[m].mean(0).min():.3f} to {C4a[m].mean(0).max():.3f} bits/s")

**Result: fig4a-coherence-after-higher.** Over the 84 Speech-R electrodes on temporal or perisylvian cortex, AFTER/MIDDLE coherence is higher than BEFORE/MIDDLE in both post-stimulus windows (0 to 1 s: +0.40 bits/s, P = 0.0023; 1 to 2.5 s: +0.29 bits/s, P = 0.0053) and not before the sound (P = 0.137). Over all 251 temporal or perisylvian electrodes only the late window differs (P = 0.032; 0 to 1 s P = 0.65). All seven subjects have a positive mean difference on set (a). Reproduced on the Speech-R set, with a multitaper bandwidth the paper does not give.

In [ ]:
rng = np.random.default_rng(405)
rows = []
for sub in ["all", "condition"]:
    C4b, _ = coherence_bits(sub)
    for r in sum([window_tests(C4b, m, rng, k) for k, m in sets4.items()], []):
        r["subtracted"] = "(a) mean over all trials" if sub == "all" else "(b) mean per condition"
        rows.append(r)
R4b = pd.DataFrame(rows)
print(R4b.to_string())

**Result: fig4b-coherence-sentence-specific.** After subtracting the mean time course, the 1 to 2.5 s window still shows higher AFTER/MIDDLE coherence on the Speech-R set, with the mean over all trials subtracted (+0.20 bits/s, P = 0.038) and with the mean per condition (+0.29, P = 0.0025). Unlike the paper (P = 0.241), the 0 to 1 s window is also significant here (P = 0.0057 and 0.0056). On all temporal or perisylvian electrodes the late window gives P = 0.106 and 0.016. Reproduced on the Speech-R set.

## eSTRF fits

Methods, eSTRF model formulation: ridge regression of HFB on the auditory spectrogram at lags 0
to 400 ms, inputs and outputs z-scored (with the training trials' statistics here), one trial
left out per split, R2 on the held-out trial; the ridge parameter is the mode of the per-split
best values over active (Speech-R) electrodes and all conditions, and every model is re-fit with
it; coefficients averaged over splits, and their mean over s.d. across splits where the paper
says so. The ridge grid is a standard choice (log-spaced, 1 to 1e8), checked below to have its
mode inside the grid. Samples per trial (claims, open detail 7): (a) onset to offset, the main
reading; (b) onset to 400 ms after offset, run for the selection counts.

In [ ]:
ALPHAS = np.logspace(0, 8, 17)
def cond_idx(tr, phase, noise=False, filt=None):
    sel = (tr.trial_num >= 0) & (tr.noise == noise) & (tr.trial_phase == phase)
    if filt is not None:
        sel &= tr.filt_type == filt
    return np.where(sel.to_numpy())[0]

t0 = time.time()
best, best_el = [], []
for s, d in D.items():
    sr = E.loc[E.subject.eq(s), "speech_r"].to_numpy()
    for ph in ["bef", "mid", "aft"]:
        Xs, ys = ac4.trial_design(d, cond_idx(d["trials"], ph))
        _, r2 = ac4.ridge_loto(Xs, ys, ALPHAS)
        best.append(ALPHAS[r2.argmax(1)][:, sr].ravel())
        best_el.append(ALPHAS[r2.mean(0).argmax(0)][sr])
best = np.concatenate(best)
vals, counts = np.unique(best, return_counts=True)
print(pd.Series(counts, index=[f"{v:.0e}" for v in vals]).to_string())
print(f"mode of all split values {vals[np.argmax(counts)]:.0e}, the top of the grid: in these splits no alpha predicts the held-out trial better than the null model")
inner = best[best < ALPHAS[-1]]
vals_i, counts_i = np.unique(inner, return_counts=True)
ALPHA = vals_i[np.argmax(counts_i)]
v_el, c_el = np.unique(np.concatenate(best_el), return_counts=True)
print(f"adopted: mode over split values below the top of the grid {ALPHA:.0e} ({len(inner)} of {len(best)} values)")
print(f"variant: mode of each electrode's best alpha (mean R2 over splits) {v_el[np.argmax(c_el)]:.0e}")
print(f"{time.time() - t0:.0f} s")

FITS = {}
def fit(s, key, idx, window="sound"):
    d = D[s]
    Xs, ys = ac4.trial_design(d, idx, window)
    coefs, r2 = ac4.ridge_loto(Xs, ys, [ALPHA])
    coefs, r2 = coefs[:, 0], r2[:, 0]
    FITS[(s, key)] = dict(mean=coefs.mean(0), z=coefs.mean(0) / coefs.std(0), r2=r2, n_trials=len(idx))
t0 = time.time()
for s, d in D.items():
    tr = d["trials"]
    for ph in ["bef", "mid", "aft"]:
        fit(s, ph, cond_idx(tr, ph))
    fit(s, "mid_plus", cond_idx(tr, "mid"), "plus")
    for ph in ["bef", "aft"]:
        for fl in ["temp", "freq"]:
            fit(s, f"{ph}_{fl}", cond_idx(tr, ph, filt=fl))
    if tr.noise.any():
        for ph in ["bef", "mid", "aft"]:
            fit(s, f"noise_{ph}", cond_idx(tr, ph, noise=True))
print(f"fits done in {time.time() - t0:.0f} s")
print(pd.DataFrame([dict(subject=s, fit=k, trials=v["n_trials"], median_r2=np.median(v["r2"])) for (s, k), v in FITS.items()]).pivot(index="subject", columns="fit", values="trials").to_string())

## STRF-R electrodes

> strf-r-count: 53 of 468 electrodes (11.3%) were well modelled by the eSTRF (STRF-R), generally
> on perisylvian temporal lobe.

Methods, STRF-responsive electrode selection: the lower bound of the R2 over cross-validation
splits of the MIDDLE model above 0. Readings (claims, open detail 8): (a) 0.5th percentile of a
bootstrap of the mean R2 over splits, (b) 1st percentile of the per-split R2, (c) 0.5th
percentile of the per-split R2; with the samples of open detail 7, (a) and (b).

In [ ]:
rng = np.random.default_rng(606)
rows = []
for key, wlab in [("mid", "(a) onset-offset"), ("mid_plus", "(b) to offset+400 ms")]:
    lb = {"a": [], "b": [], "c": []}
    for s in D:
        r2 = FITS[(s, key)]["r2"]
        lb["a"].append(ac4.bootstrap_lower(r2, rng))
        lb["b"].append(np.percentile(r2, 1, axis=0))
        lb["c"].append(np.percentile(r2, 0.5, axis=0))
    for k in lb:
        v = np.concatenate(lb[k])
        E[f"strf_{key}_{k}"] = v > 0
        rows.append(dict(samples=wlab, bound=k, strf_r=int((v > 0).sum()), within_53_tol=abs((v > 0).sum() - 53) <= 5.3,
                         **{s: int((v[E.subject.eq(s).to_numpy()] > 0).sum()) for s in D}))
R_strf = pd.DataFrame(rows)
print(R_strf.to_string())
E["mid_r2"] = np.concatenate([FITS[(s, "mid")]["r2"].mean(0) for s in D])
E["strf_r"] = E["strf_mid_a"]
print("adopted: samples (a), bound (a); STRF-R", int(E.strf_r.sum()))
print(pd.crosstab(E.region, E.strf_r))
print(f"mean held-out R2 of MIDDLE models over STRF-R electrodes: {E.loc[E.strf_r, 'mid_r2'].mean():.3f} (range {E.loc[E.strf_r, 'mid_r2'].min():.3f} to {E.loc[E.strf_r, 'mid_r2'].max():.3f})")

**Result: strf-r-count.** The MIDDLE eSTRFs predict held-out trials in few electrodes: 9 electrodes are STRF-R under the bootstrap bound (13 with samples to 400 ms after offset), and 0 or 1 under the per-split percentiles, against the paper's 53. Their mean held-out R2 ranges from 0.044 to 0.226, and five subjects contribute at most one electrode each. Not reproduced.

## Electrodes included in the eSTRF analyses

> included-electrode-count: 41 of 468 electrodes (8.76%) were Speech-R, STRF-R and on the temporal
> lobe or perisylvian cortex, and enter the eSTRF analyses.

Location from the anatomical label of each channel in `channels.tsv` (claims, open detail 6):
temporal lobe (`temporal-*`, `sub_temporal`) and perisylvian (`*-inferior` frontal, premotor,
motor and parietal). The paper's Figure 10c legend restricts the model results to the temporal
lobe; that set is shown too.

In [ ]:
rows = []
for zr in R_speech.itertuples():
    sr = E[f"lb_{zr.zscore}_{zr.window}_{zr.trials}"] > 0
    for key in ["mid_a", "mid_b", "mid_c", "mid_plus_a", "mid_plus_b", "mid_plus_c"]:
        st = E[f"strf_{key}"]
        rows.append(dict(speech_r=f"{zr.zscore}/{zr.window}/{zr.trials}", strf=key,
                         temporal_perisylvian=int((sr & st & E.region.isin(["temporal", "perisylvian"])).sum()),
                         temporal_only=int((sr & st & E.region.eq("temporal")).sum())))
R_inc = pd.DataFrame(rows)
R_inc["within_41_tol"] = (R_inc.temporal_perisylvian - 41).abs() <= 4.1
print(R_inc.to_string())
print("readings within 41 +/- 4.1:", int(R_inc.within_41_tol.sum()), "of", len(R_inc))
E["included"] = E.speech_r & E.strf_r & E.region.isin(["temporal", "perisylvian"])
print("adopted reading: included", int(E.included.sum()))
print(E[E.included].groupby(["subject", "region"]).size().unstack(fill_value=0))

**Result: included-electrode-count.** The STRF-R electrodes are all Speech-R and on temporal or perisylvian cortex, so the included count follows the STRF-R count: 9 under the adopted readings and at most 13 under any, against 41. They come from three subjects (S01, S02, S04); none from the pink noise subjects. Not reproduced.

### Pink noise HFB on the included electrodes (supp3a, set (a))

In [ ]:
m_inc_n = E.loc[E.subject.isin(noise_subjects), "included"].to_numpy()
print("included electrodes in S07, S08, S13:", int(m_inc_n.sum()))
if m_inc_n.sum() >= 2:
    print(cluster_report(DIFN[m_inc_n], "(a) included electrodes of S07, S08, S13", 5, ttn).to_string())
    print(f"pink noise AFTER-BEFORE mean over (a), 0-3 s: {DIFN[m_inc_n][:, ttn >= 0].mean():.3f} z (n = {m_inc_n.sum()})")
else:
    print("set (a) is empty or a single electrode: no test; the claim is judged on set (b)")

**Result: supp3a-pink-noise-no-hfb-change.** On the pink noise triplets, no cluster of AFTER minus BEFORE reaches P < 0.05 over the 36 Speech-R electrodes of S07, S08 and S13 (smallest P = 0.0895), so the claim of absence is reproduced on set (b). Set (a) is empty: none of these subjects' electrodes is included in the eSTRF analyses here. The mean AFTER minus BEFORE difference over 0 to 3 s is the same in pink noise and speech triplets (0.041 z in both), and the speech triplets of the same electrodes do give significant clusters (P = 0.0013), so the absence here reflects less power more than a smaller effect.

## MTF of eSTRFs against filter type

> fig8-mtf-matches-filter: the MTFs of the eSTRFs match the filter type of the stimuli they were
> fit on, in BEFORE, AFTER and in the AFTER minus BEFORE shift.

eSTRFs fit apart on temporal-filtered and spectral-filtered triplets (Figure 8 legend), the MTF
as the amplitude of the 2-D FFT of the mean eSTRF (32 bands x 21 lags), the temporal minus
spectral MTF difference per included electrode, and MNE's cluster permutation test over the
modulation plane (the half plane of non-negative spectral modulation; the amplitude spectrum of a
real eSTRF is symmetric). The criterion's region test: a positive cluster (temporal-filter MTF
higher) must lie at low temporal and high spectral modulation, a negative one at high temporal
and low spectral modulation; "low" and "high" are taken here as below and above the median of
the plane's absolute modulations, measured at the cluster's centre (mean over its bins).

In [ ]:
inc = E.included.to_numpy()
def strf_of(s, key, ch):
    return FITS[(s, key)]["mean"][:, ch]
tm = np.fft.fftshift(np.fft.fftfreq(ac4.MAX_LAG + 1, 1 / ac4.FS_OUT))      # Hz
sm = np.fft.fftshift(np.fft.fftfreq(ac4.N_BANDS, 1.0))                     # cycles per band
half = sm >= 0
def mtf_diffs(kind):
    out = []
    for s in D:
        chs = np.where(E.loc[E.subject.eq(s), "included"].to_numpy())[0]
        for c in chs:
            if kind in ("bef", "aft"):
                mt = ac4.mtf(strf_of(s, f"{kind}_temp", c)); ms = ac4.mtf(strf_of(s, f"{kind}_freq", c))
            elif kind == "shift_a":    # MTF of the AFTER minus BEFORE eSTRF
                mt = ac4.mtf(strf_of(s, "aft_temp", c) - strf_of(s, "bef_temp", c))
                ms = ac4.mtf(strf_of(s, "aft_freq", c) - strf_of(s, "bef_freq", c))
            else:                      # difference of MTFs
                mt = ac4.mtf(strf_of(s, "aft_temp", c)) - ac4.mtf(strf_of(s, "bef_temp", c))
                ms = ac4.mtf(strf_of(s, "aft_freq", c)) - ac4.mtf(strf_of(s, "bef_freq", c))
            out.append((mt - ms)[half])
    return np.array(out)
TMg, SMg = np.meshgrid(np.abs(tm), sm[half])
med_t, med_s = np.median(TMg), np.median(SMg)
rows = []
for kind in ["bef", "aft", "shift_a", "shift_b"]:
    X = mtf_diffs(kind)
    T, clusters, pv = clusters_of(X, 8)
    for c, p in zip(clusters, pv):
        sign = "+" if T[c].mean() > 0 else "-"
        ct, cs = TMg[c].mean(), SMg[c].mean()
        match = (sign == "+" and ct < med_t and cs > med_s) or (sign == "-" and ct > med_t and cs < med_s)
        rows.append(dict(kind=kind, n=len(X), p=p, sign=sign, bins=int(c.sum()), centre_temporal_Hz=ct,
                         centre_spectral_cyc_per_band=cs, in_filter_region=match))
    if not len(clusters):
        rows.append(dict(kind=kind, n=len(X), p=np.nan, sign="none"))
R8 = pd.DataFrame(rows)
print(f"median |temporal modulation| {med_t:.2f} Hz, median spectral modulation {med_s:.3f} cycles/band")
print(R8.sort_values(["kind", "p"]).to_string())

**Result: fig8-mtf-matches-filter.** With 9 electrodes, the temporal minus spectral MTF difference has a positive cluster in the region the temporal filter keeps (low temporal, high spectral modulation) in BEFORE (centre 2.1 Hz, P = 0.0078) and in AFTER (centre 2.0 Hz, P = 0.043). The larger positive clusters lie at high temporal modulations (16 to 19 Hz, P = 0.0039 to 0.031), outside both filter regions. For the AFTER minus BEFORE shift, the only significant cluster (P = 0.0039) lies at 15.3 Hz, outside the regions, with the MTF of the difference, and no cluster reaches P < 0.05 with the difference of MTFs. The match holds in BEFORE and AFTER and not in the shift: partially reproduced, on far fewer electrodes than the paper's 41.

## eSTRF output power to unfiltered speech

> fig9-rms-increase: Filtered speech eSTRFs from AFTER give larger predicted responses to
> unfiltered speech than those from BEFORE (mean RMS increase 0.12 plus/minus 0.03, P = 0.0001,
> n = 41).

Methods, eSTRF unfiltered speech output power analysis: each eSTRF standardized to zero mean and
unit s.d., the subject's MIDDLE spectrograms (z-scored as model inputs) passed through the BEFORE
and the AFTER eSTRF, and the RMS of the output compared. Readings: the eSTRF as (a) mean
coefficients over splits or (b) their z-score over splits; the RMS (a) over all MIDDLE sounds
concatenated or (b) per sound then averaged.

In [ ]:
rng = np.random.default_rng(909)
rows = []
for strf_kind in ["mean", "z"]:
    for rms_kind in ["concat", "per_sound"]:
        diffs, befs = [], []
        for s, d in D.items():
            chs = np.where(E.loc[E.subject.eq(s), "included"].to_numpy())[0]
            if not len(chs):
                continue
            Xs, _ = ac4.trial_design(d, cond_idx(d["trials"], "mid"))
            Xa = np.concatenate(Xs)
            mu, sd = Xa.mean(0), Xa.std(0)
            for c in chs:
                v = {}
                for ph in ["bef", "aft"]:
                    w = FITS[(s, ph)][strf_kind][:, c]
                    w = (w - w.mean()) / w.std()
                    if rms_kind == "concat":
                        v[ph] = np.sqrt(np.mean((((Xa - mu) / sd) @ w) ** 2))
                    else:
                        v[ph] = np.mean([np.sqrt(np.mean((((X - mu) / sd) @ w) ** 2)) for X in Xs])
                diffs.append(v["aft"] - v["bef"]); befs.append(v["bef"])
        diffs, befs = np.array(diffs), np.array(befs)
        obs, p = ac4.signflip_test(diffs, rng)
        rows.append(dict(strf=strf_kind, rms=rms_kind, n=len(diffs), rms_before=befs.mean(), relative_increase=(diffs / befs).mean(), mean_increase=obs,
                         se=diffs.std(ddof=1) / np.sqrt(len(diffs)), sd=diffs.std(ddof=1), p=p,
                         within_tol=abs(obs - 0.12) <= 0.03, frac_positive=(diffs > 0).mean()))
R9a = pd.DataFrame(rows)
print(R9a.to_string())

**Result: fig9-rms-increase.** Over the 9 included electrodes, the AFTER eSTRF gives a larger RMS output to unfiltered speech than the BEFORE eSTRF under every reading (mean increase 11.4 to 12.0, 8 of 9 electrodes positive with mean coefficients), with P = 0.034 and 0.031 for mean coefficients and 0.077 and 0.073 for z-scored coefficients. The RMS is about 105 here, against about 4 on the paper's axis, so the increase cannot be compared with 0.12 on the paper's scale; as a fraction of the BEFORE RMS it is 0.12 to 0.13. The direction holds; the stated value does not: partially reproduced.

## MIDDLE eSTRFs predict AFTER better than BEFORE

> fig9-middle-generalizes-after: eSTRFs fit on unfiltered speech predict HFB activity better in
> AFTER than in BEFORE (mean R2 improvement 0.05 plus/minus 0.01, P = 0.0001, n = 41).

Methods, MIDDLE condition coefficients generalization: the MIDDLE eSTRF (mean coefficients)
convolved with the BEFORE and AFTER spectrograms, and R2 against the HFB. Spectrograms and HFB of
each condition are z-scored by that condition's own statistics (Methods: inputs and outputs
z-scored). R2 (a) per trial then averaged, (b) over all trials concatenated.

In [ ]:
rng = np.random.default_rng(910)
rows = []
for r2_kind in ["per_trial", "concat"]:
    diffs, vals = [], []
    for s, d in D.items():
        chs = np.where(E.loc[E.subject.eq(s), "included"].to_numpy())[0]
        if not len(chs):
            continue
        W = FITS[(s, "mid")]["mean"][:, chs]
        r = {}
        for ph in ["bef", "aft"]:
            Xs, ys = ac4.trial_design(d, cond_idx(d["trials"], ph))
            Xa, ya = np.concatenate(Xs), np.concatenate(ys)[:, chs]
            mu, sd, my, sy = Xa.mean(0), Xa.std(0), ya.mean(0), ya.std(0)
            if r2_kind == "concat":
                pred = ((Xa - mu) / sd) @ W
                yz = (ya - my) / sy
                r[ph] = 1 - ((yz - pred) ** 2).sum(0) / ((yz - yz.mean(0)) ** 2).sum(0)
            else:
                rr = []
                for X, y in zip(Xs, ys):
                    pred = ((X - mu) / sd) @ W
                    yz = (y[:, chs] - my) / sy
                    rr.append(1 - ((yz - pred) ** 2).sum(0) / ((yz - yz.mean(0)) ** 2).sum(0))
                r[ph] = np.mean(rr, 0)
        diffs.append(r["aft"] - r["bef"]); vals.append(np.column_stack([r["bef"], r["aft"]]))
    diffs = np.concatenate(diffs); vals = np.concatenate(vals)
    obs, p = ac4.signflip_test(diffs, rng)
    rows.append(dict(r2=r2_kind, n=len(diffs), r2_before=vals[:, 0].mean(), r2_after=vals[:, 1].mean(),
                     mean_improvement=obs, se=diffs.std(ddof=1) / np.sqrt(len(diffs)), sd=diffs.std(ddof=1),
                     p=p, within_tol=abs(obs - 0.05) <= 0.01, frac_positive=(diffs > 0).mean()))
R9b = pd.DataFrame(rows)
print(R9b.to_string())

**Result: fig9-middle-generalizes-after.** Over the 9 included electrodes, the MIDDLE eSTRF predicts AFTER better than BEFORE: with R2 over concatenated trials the improvement is 0.051 (P = 0.021), within 0.05 +/- 0.01; with R2 per trial it is 0.089 (P = 0.020). Reproduced under reading (b), on 9 electrodes against the paper's 41.

## Partial correlation of eSTRFs

> fig10-partial-correlation: Partial correlations of eSTRFs between AFTER and MIDDLE (controlling
> for BEFORE) exceed those between BEFORE and MIDDLE (controlling for AFTER), a mean improvement
> of 0.18 (P = 0.001, n = 41).

Methods, eSTRF linear overlap partial correlation analysis: each eSTRF as the z-score of each
coefficient across CV splits; sim(aft, mid) = pcorr(after, middle | before), sim(bef, mid) =
pcorr(before, middle | after); sign-flip test of the difference. Regression without (a) and with
(b) an intercept.

In [ ]:
rng = np.random.default_rng(1010)
def pcorr_table(keys=("bef", "mid", "aft"), mask_col="included", subjects=None):
    rows = []
    for s in (subjects or list(D)):
        chs = np.where(E.loc[E.subject.eq(s), mask_col].to_numpy())[0]
        for c in chs:
            b, m, a = (FITS[(s, k)]["z"][:, c] for k in keys)
            row = dict(subject=s, ch=c)
            for icpt in [False, True]:
                tag = "b" if icpt else "a"
                row[f"aft_mid_{tag}"] = ac4.partial_corr(a, m, b, icpt)
                row[f"bef_mid_{tag}"] = ac4.partial_corr(b, m, a, icpt)
                row[f"diff_{tag}"] = row[f"aft_mid_{tag}"] - row[f"bef_mid_{tag}"]
            rows.append(row)
    return pd.DataFrame(rows)
PC = pcorr_table()
rows = []
for tag in ["a", "b"]:
    obs, p = ac4.signflip_test(PC[f"diff_{tag}"], rng)
    rows.append(dict(intercept=tag, n=len(PC), bef_mid=PC[f"bef_mid_{tag}"].mean(), aft_mid=PC[f"aft_mid_{tag}"].mean(),
                     mean_diff=obs, median_diff=PC[f"diff_{tag}"].median(), se=PC[f"diff_{tag}"].std(ddof=1) / np.sqrt(len(PC)),
                     p=p, within_tol=abs(obs - 0.18) <= 0.018, frac_positive=(PC[f"diff_{tag}"] > 0).mean()))
R10 = pd.DataFrame(rows)
print(R10.to_string())
print(PC.groupby("subject")[["diff_a"]].agg(["count", "mean"]))
print("temporal-lobe electrodes only (Figure 10c legend):")
PCt = PC.merge(E.reset_index()[["subject", "region"]].assign(ch=E.groupby("subject").cumcount().to_numpy()), on=["subject", "ch"])
obs, p = ac4.signflip_test(PCt.loc[PCt.region.eq("temporal"), "diff_a"], rng)
print(f"  n = {PCt.region.eq('temporal').sum()}, mean difference {obs:.3f}, P = {p:.4f}")

**Result: fig10-partial-correlation.** Over the 9 included electrodes, AFTER/MIDDLE partial correlation exceeds BEFORE/MIDDLE by a mean 0.308 (P = 0.011) without an intercept and 0.305 (P = 0.0042) with one; 8 of 9 electrodes are positive. The mean is outside 0.18 +/- 0.018 (the median is 0.16). The effect is present and larger: partially reproduced.

> fig10b-similarity-correlates: the increase in eSTRF similarity correlates with the electrode's
> evoked HFB amplitude or model score (Pearson r = 0.42, P = 0.007).

In [ ]:
Einc = E[E.included].copy()
Einc["ch"] = E.groupby("subject").cumcount()[E.included].to_numpy()
Einc["evoked"] = [POST[(s, MAIN[0], MAIN[1])][speech(D[s]["trials"]).to_numpy() & (D[s]["trials"].trial_phase == "mid").to_numpy()][:, c].mean()
                  for s, c in zip(Einc.subject, Einc.ch)]
M = PC.merge(Einc[["subject", "ch", "evoked", "mid_r2", "description"]], on=["subject", "ch"])
for x in ["evoked", "mid_r2"]:
    for tag in ["a", "b"]:
        r, p = stats.pearsonr(M[x], M[f"diff_{tag}"])
        print(f"({'a' if x == 'evoked' else 'b'}) {x} vs partial correlation difference (intercept {tag}): r = {r:.3f}, P = {p:.4f}, n = {len(M)}, within 0.42 +/- 0.042: {abs(r - 0.42) <= 0.042}")
print(f"evoked range {M.evoked.min():.2f} to {M.evoked.max():.2f} z; mid_r2 range {M.mid_r2.min():.3f} to {M.mid_r2.max():.3f}")

**Result: fig10b-similarity-correlates.** Across the 9 included electrodes, the partial correlation difference correlates with the mean evoked HFB at r = 0.63 (P = 0.072) and with the MIDDLE model score at r = 0.55 (P = 0.126). Neither r is within 0.42 +/- 0.042 and neither P is below 0.05. Not reproduced.

> fig10c-stg-location: Most of the increase in partial correlation lies around the superior
> temporal gyrus.

STG electrodes are those labelled `temporal-superior` in `channels.tsv`.

In [ ]:
M["stg"] = M.description.eq("temporal-superior")
pos = M[M.diff_a > 0]
print("included electrodes by label:", M.description.value_counts().to_dict())
print(f"(a) share of the summed positive differences on STG: {pos.loc[pos.stg, 'diff_a'].sum() / pos.diff_a.sum():.3f} ({pos.loc[pos.stg, 'diff_a'].sum():.2f} of {pos.diff_a.sum():.2f})")
print(f"(b) share of electrodes with a positive difference on STG: {pos.stg.mean():.3f} ({int(pos.stg.sum())} of {len(pos)})")
print(f"reference: share of included electrodes on STG {M.stg.mean():.3f} ({int(M.stg.sum())} of {len(M)})")

**Result: fig10c-stg-location.** Of the summed positive partial correlation differences, 51.7% lie on the 4 electrodes labelled temporal-superior (reading a), and 4 of 8 electrodes with a positive difference are STG (50%, reading b, not more than half). Reading (a) holds, by a small margin on 9 electrodes: reproduced.

> supp3b-pink-noise-no-pcorr-change: In pink noise trials the partial correlations of
> BEFORE/MIDDLE and AFTER/MIDDLE eSTRFs do not differ (permutation test, n = 12).

eSTRFs fit on the pink noise triplets (the MIDDLE eSTRF on the pink noise), with the same ridge
parameter, for the included electrodes of S07, S08 and S13.

In [ ]:
rng = np.random.default_rng(303)
PCn = pcorr_table(keys=("noise_bef", "noise_mid", "noise_aft"), subjects=noise_subjects)
print("included electrodes in S07, S08, S13:", len(PCn), "(paper: 12)")
for tag in (["a", "b"] if len(PCn) >= 2 else []):
    obs, p = ac4.signflip_test(PCn[f"diff_{tag}"], rng)
    print(f"intercept {tag}: n = {len(PCn)}, mean AFTER/MIDDLE {PCn[f'aft_mid_{tag}'].mean():.3f}, BEFORE/MIDDLE {PCn[f'bef_mid_{tag}'].mean():.3f}, difference {obs:.3f}, P = {p:.4f}")
if len(PCn):
    print(PCn.groupby("subject").size().to_string())
    print(f"MIDDLE pink noise model R2, median over these electrodes: {np.median([FITS[(s, 'noise_mid')]['r2'].mean(0)[c] for s, c in zip(PCn.subject, PCn.ch)]):.3f}")

**Result: supp3b-pink-noise-no-pcorr-change.** No electrode of S07, S08 or S13 is included in the eSTRF analyses here, so there is nothing to test; the claim is blocked.

## Connectivity

> supp6a-connectivity-increase: HFB amplitude coherence between eSTRF seed electrodes and frontal
> and temporal target electrodes rises slightly in AFTER against BEFORE (temporal-frontal p =
> 0.025, temporal-temporal p = 0.043, n = 39).

Supplementary Methods, Inter- and Intra-regional Coherence Analysis: seeds are the included
electrodes on the temporal lobe; targets all other temporal electrodes, or the frontal/premotor
electrodes (`frontal-*`, `premotor-*` labels); per trial the multitaper coherence of the HFB
amplitude ("similar to that described above": the 400 ms windows of Figure 4, here the
post-stimulus windows 0 to 2.5 s), averaged over targets, then over trials, integrated over
frequency in bits/s, AFTER minus BEFORE per seed, and MNE's permutation t-test across seeds.

In [ ]:
def seed_coherence(target_region):
    rows = []
    for s, d in D.items():
        g = E[E.subject.eq(s)].reset_index(drop=True)
        seeds = np.where((g.included & g.region.eq("temporal")).to_numpy())[0]
        if target_region == "frontal":
            targ_all = np.where(g.description.str.startswith(("frontal", "premotor")).to_numpy())[0]
        else:
            targ_all = np.where(g.region.eq("temporal").to_numpy())[0]
        ep, times = EP[s]
        zep = ac4.zscore_epochs(d, ep, times, MAIN[0])
        tr = d["trials"]
        for c in seeds:
            targ = targ_all[targ_all != c]
            if not len(targ):
                continue
            row = dict(subject=s, seed=c, n_targets=len(targ))
            for ph in ["bef", "mid", "aft"]:
                X = zep[cond_idx(tr, ph)]
                bits = []
                for ws in WIN_STARTS[WIN_STARTS >= 0]:
                    sel = (times >= ws - 1e-9) & (times < ws + 0.4 - 1e-9)
                    f, coh = ac4.mt_coherence(X[:, [c]][..., sel], X[:, targ][..., sel], ac4.FS_OUT, nw=2.0)
                    bits.append(ac4.coh_to_bits(coh.mean(1), f).mean())   # mean over targets, then trials
                row[ph] = np.mean(bits)
            rows.append(row)
    return pd.DataFrame(rows)
COH6 = {}
for reg in ["frontal", "temporal"]:
    t = seed_coherence(reg)
    COH6[reg] = t
    diff = (t.aft - t.bef).to_numpy()
    T, p, _ = mne.stats.permutation_t_test(diff[:, None], n_permutations=10000, seed=6, tail=0)
    print(f"temporal seeds to {reg} targets: seeds {len(t)} (subjects {t.subject.nunique()}), BEFORE {t.bef.mean():.3f}, MIDDLE {t.mid.mean():.3f}, AFTER {t.aft.mean():.3f} bits/s; AFTER-BEFORE mean {diff.mean():.4f}, t = {T[0]:.2f}, P = {p[0]:.4f}")
included_temporal = int((E.included & E.region.eq("temporal")).sum())
print("included temporal seeds:", included_temporal, "; seeds with frontal targets:", len(COH6["frontal"]))

**Result: supp6a-connectivity-increase.** With the 6 included temporal electrodes as seeds (three subjects), HFB coherence with frontal/premotor targets rises from BEFORE to AFTER (+0.173 bits/s, t = 3.24, P = 0.031), and with temporal targets rises without significance (+0.110, t = 2.05, P = 0.094). The frontal part is reproduced and the temporal part not: partially reproduced.

> supp6a-after-closer-to-middle: the seed to target coherence in AFTER is closer than in BEFORE to
> that obtained with unfiltered speech. Measure (criterion): the mean over seeds of |AFTER minus
> MIDDLE| against that of |BEFORE minus MIDDLE|.

In [ ]:
for reg, t in COH6.items():
    da, db = (t.aft - t.mid).abs().mean(), (t.bef - t.mid).abs().mean()
    print(f"{reg} targets: mean |AFTER-MIDDLE| {da:.4f}, mean |BEFORE-MIDDLE| {db:.4f}, AFTER closer: {da < db}; means BEFORE {t.bef.mean():.3f}, MIDDLE {t.mid.mean():.3f}, AFTER {t.aft.mean():.3f}")

**Result: supp6a-after-closer-to-middle.** For both target regions the AFTER coherence is closer to MIDDLE than BEFORE is: mean |AFTER - MIDDLE| 0.089 against |BEFORE - MIDDLE| 0.156 bits/s for frontal targets, and 0.207 against 0.214 for temporal targets (6 seeds). Reproduced under the criterion's measure, by a small margin for temporal targets.

> supp6b-no-pac-change: Phase-amplitude coupling between 3 to 8 Hz phase in frontal/premotor
> electrodes and HFB amplitude in the eSTRF electrodes does not change from BEFORE to AFTER.

Supplementary Methods, Theta Connectivity Phase-Amplitude Coupling Analysis: phase from 3 to 10
Hz in 0.5 Hz steps (band-pass of +/- 1 Hz around each, a standard choice) on the frontal/premotor
electrodes, HFB amplitude from 10 log-spaced bands 70 to 140 Hz on the included electrodes, the
post-stimulus samples (onset to sound end) concatenated over trials per condition, PAC as in
Ozkurt and Schnitzler (2011), averaged over phases 3 to 8 Hz (variants 4 to 8 and 2 to 12 Hz,
the legend's), then over phase electrodes; AFTER minus BEFORE; permutation t-test against 0.

In [ ]:
PH_F = np.arange(2.0, 12.01, 0.5)
def ozkurt(amp, ph):
    n = amp.shape[-1]
    return np.abs((amp * np.exp(1j * ph)).sum(-1)) / (np.sqrt(n) * np.sqrt((amp ** 2).sum(-1)))
rows = []
for s, d in D.items():
    g = E[E.subject.eq(s)].reset_index(drop=True)
    amp_ch = np.where(g.included.to_numpy())[0]
    ph_ch = np.where(g.description.str.startswith(("frontal", "premotor")).to_numpy())[0]
    if not len(amp_ch) or not len(ph_ch):
        continue
    tr = d["trials"]
    t_ep = np.arange(int(ac4.EPOCH[0] * ac4.FS), int(ac4.EPOCH[1] * ac4.FS)) / ac4.FS
    res = {}
    for ph in ["bef", "aft"]:
        idx = cond_idx(tr, ph)
        raw = d["raw_ep"][idx].astype(np.float64)
        amp = ac4.hfb_amplitude(raw[:, amp_ch], np.geomspace(70, 140, 10))
        phases = np.stack([ac4.band_phase(raw[:, ph_ch], f - 1, f + 1) for f in PH_F])  # (freq, trials, ch, t)
        masks = [(t_ep >= 0) & (t_ep < dur) for dur in tr.duration.to_numpy()[idx]]
        A = np.concatenate([amp[i][:, m] for i, m in enumerate(masks)], axis=-1)          # (amp_ch, samples)
        P = np.concatenate([phases[:, i][..., m] for i, m in enumerate(masks)], axis=-1)  # (freq, ph_ch, samples)
        res[ph] = ozkurt(A[None, None], P[:, :, None])                                    # (freq, ph_ch, amp_ch)
    for j, c in enumerate(amp_ch):
        row = dict(subject=s, ch=c, n_phase_ch=len(ph_ch))
        for lab, lo, hi in [("3-8", 3, 8), ("4-8", 4, 8), ("2-12", 2, 12)]:
            fm = (PH_F >= lo) & (PH_F <= hi)
            row[f"bef_{lab}"] = res["bef"][fm][:, :, j].mean()
            row[f"aft_{lab}"] = res["aft"][fm][:, :, j].mean()
        rows.append(row)
PAC = pd.DataFrame(rows)
for lab in ["3-8", "4-8", "2-12"]:
    diff = (PAC[f"aft_{lab}"] - PAC[f"bef_{lab}"]).to_numpy()
    T, p, _ = mne.stats.permutation_t_test(diff[:, None], n_permutations=10000, seed=7, tail=0)
    print(f"phase {lab} Hz: n = {len(diff)} amplitude electrodes ({PAC.subject.nunique()} subjects), BEFORE {PAC[f'bef_{lab}'].mean():.4f}, AFTER {PAC[f'aft_{lab}'].mean():.4f}, AFTER-BEFORE {diff.mean():.5f}, t = {T[0]:.2f}, P = {p[0]:.4f}")

**Result: supp6b-no-pac-change.** PAC between 3 to 8 Hz frontal/premotor phase and HFB amplitude of the 9 included electrodes does not change from BEFORE to AFTER (difference 0.00058, t = 1.63, P = 0.141); the variants give P = 0.055 (4 to 8 Hz) and 0.54 (2 to 12 Hz). Reproduced.

## Coherence with the speech envelope

> supp7-envelope-coherence: In temporal electrodes, theta coherence with the speech envelope is
> higher for unfiltered speech than for BEFORE (p = 0.001, n = 72), but does not differ between
> BEFORE and AFTER; frontal electrodes show no effects.

Supplementary Methods and Supplementary Figure 7: the speech envelope from 64 log-spaced
wavelets 500 to 2000 Hz, rectified and averaged; multitaper coherence of the re-referenced ECoG
with the envelope over 0 to 3 s per trial (both resampled to 100 Hz; NW = 3, five tapers,
jackknife, a standard choice), converted to normal mutual information per frequency
(-log2(1 - coh)), averaged over trials; the MIDDLE minus BEFORE and AFTER minus BEFORE differences
per frequency, and MNE's cluster permutation test over frequency (1 to 30 Hz) across the Speech-R
electrodes of the temporal lobe and of frontal/premotor cortex.

In [ ]:
import scipy.signal as ss
NMI = []
for s, d in D.items():
    tr = d["trials"]
    t_ep = np.arange(int(ac4.EPOCH[0] * ac4.FS), int(ac4.EPOCH[1] * ac4.FS)) / ac4.FS
    sel = (t_ep >= 0) & (t_ep < 3.0)
    out = {}
    for ph in ["bef", "mid", "aft"]:
        idx = cond_idx(tr, ph)
        x = ss.resample_poly(d["raw_ep"][idx][..., sel].astype(np.float64), 1, 10, axis=-1)       # (trials, ch, 300)
        env = np.stack([ss.resample_poly(d["env"][i][:3000], 1, 10) for i in idx])                # (trials, 300)
        f, coh = ac4.mt_coherence(x, np.broadcast_to(env[:, None], x.shape), 100.0, nw=3.0)
        out[ph] = (-np.log2(1 - np.clip(coh, 0, 1 - 1e-9))).mean(0)                                # (ch, freqs)
    NMI.append(out)
NMI = {ph: np.concatenate([o[ph] for o in NMI]) for ph in ["bef", "mid", "aft"]}
fsel = (f >= 1) & (f <= 30)
print("frequencies tested:", f[fsel])
rows = []
groups = {"temporal": (E.speech_r & E.region.eq("temporal")).to_numpy(),
          "frontal": (E.speech_r & E.description.str.startswith(("frontal", "premotor"))).to_numpy()}
for gname, m in groups.items():
    for comp in ["mid", "aft"]:
        X = (NMI[comp] - NMI["bef"])[m][:, fsel]
        if len(X) < 3:
            rows.append(dict(electrodes=gname, comparison=f"{comp}-bef", n=len(X), p=np.nan, sign="too few"))
            continue
        T, clusters, pv = clusters_of(X, 77)
        if not len(clusters):
            rows.append(dict(electrodes=gname, comparison=f"{comp}-bef", n=len(X), p=np.nan, sign="no cluster"))
        for c, p in zip(clusters, pv):
            fc = f[fsel][c]
            rows.append(dict(electrodes=gname, comparison=f"{comp}-bef", n=len(X), p=p, sign="+" if T[c].mean() > 0 else "-",
                             f_lo=fc.min(), f_hi=fc.max(), overlaps_3_8=(fc.min() <= 8) and (fc.max() >= 3),
                             overlaps_2_12=(fc.min() <= 12) and (fc.max() >= 2)))
R7 = pd.DataFrame(rows)
print(R7.to_string())
for gname, m in groups.items():
    th = (f >= 3) & (f <= 8)
    print(f"{gname} Speech-R (n = {m.sum()}): mean NMI 3-8 Hz BEFORE {NMI['bef'][m][:, th].mean():.4f}, MIDDLE {NMI['mid'][m][:, th].mean():.4f}, AFTER {NMI['aft'][m][:, th].mean():.4f} bits")

**Result: supp7-envelope-coherence.** In the 66 temporal Speech-R electrodes, no cluster of MIDDLE minus BEFORE envelope coherence reaches P < 0.05 (the smallest, 1.7 to 2.3 Hz, P = 0.093), and none of AFTER minus BEFORE. In the 19 frontal/premotor Speech-R electrodes, MIDDLE exceeds BEFORE from 6.7 to 8.3 Hz (P = 0.0012), where the paper reports no effect. Only the absence of an AFTER minus BEFORE effect holds: not reproduced.

## HFB power by wavelets

> supp4-hfb-power-increase: Power in the HFB range rises in AFTER against BEFORE in frontal (p =
> 0.001, n = 75) and temporal (p = 0.002, n = 217) electrodes, and power below 25 Hz falls in
> frontal electrodes (p = 0.001).

Supplementary Figure 4 legend: 100 log-spaced Morlet wavelets of 5 cycles, post-stimulus power
averaged over time, AFTER minus BEFORE, cluster permutation test over frequency across frontal
and across temporal electrodes. The claims file's standard choice is 1 to 200 Hz with power in
decibels against the -800 to -100 ms baseline; the kept epochs (-1 to 3.5 s) are too short for
a 5-cycle wavelet below about 3 Hz, so the range here is 3 to 200 Hz. Post-stimulus is 0 to 2.5
s (claims, open detail 2, reading b). Frontal electrodes are the `frontal-*` and `premotor-*`
labels, temporal the temporal lobe labels; all good electrodes.

In [ ]:
WF = np.geomspace(3, 200, 100)
POW = []
for s, d in D.items():
    tr = d["trials"]
    t_ep = np.arange(int(ac4.EPOCH[0] * ac4.FS), int(ac4.EPOCH[1] * ac4.FS)) / ac4.FS
    td = t_ep[::10]
    base = (td >= -0.8) & (td <= -0.1)
    post = (td >= 0) & (td <= 2.5)
    out = {}
    for ph in ["bef", "aft"]:
        x = d["raw_ep"][cond_idx(tr, ph)].astype(np.float64)
        acc = []
        for fchunk in np.array_split(WF, 10):
            p = mne.time_frequency.tfr_array_morlet(x, ac4.FS, fchunk, n_cycles=5, output="power", decim=10)
            acc.append(10 * np.log10(p[..., post].mean(-1) / p[..., base].mean(-1)).mean(0))     # (ch, f)
        out[ph] = np.concatenate(acc, axis=1)
    POW.append(out["aft"] - out["bef"])
POW = np.concatenate(POW)
rows = []
for gname, m in [("frontal", E.description.str.startswith(("frontal", "premotor")).to_numpy()), ("temporal", E.region.eq("temporal").to_numpy())]:
    T, clusters, pv = clusters_of(POW[m], 44)
    for c, p in zip(clusters, pv):
        rows.append(dict(electrodes=gname, n=int(m.sum()), p=p, sign="+" if T[c].mean() > 0 else "-", f_lo=WF[c].min(), f_hi=WF[c].max(),
                         hfb_overlap=(WF[c].max() >= 70) and (WF[c].min() <= 150), below_25=WF[c].min() < 25))
    hf = (WF >= 70) & (WF <= 150)
    print(f"{gname}: n = {m.sum()}, mean AFTER-BEFORE 70-150 Hz {POW[m][:, hf].mean():.3f} dB, below 25 Hz {POW[m][:, WF < 25].mean():.3f} dB")
R4 = pd.DataFrame(rows)
print(R4.to_string())

**Result: supp4-hfb-power-increase.** AFTER minus BEFORE wavelet power has a positive cluster covering the HFB range in frontal/premotor electrodes (26 to 200 Hz, P = 0.0001, n = 90) and in temporal electrodes (51 to 200 Hz, P = 0.0005, n = 192), and a negative cluster below 25 Hz in frontal electrodes (4.2 to 19.4 Hz, P = 0.0001). Reproduced, with electrode counts that differ from the paper's 75 and 217.

## Summary

| Claim | Outcome |
|---|---|
| behav-no-context-baseline, behav-context-enhancement, supp1a-linguistic-context, supp1a-speaker-no-difference, supp1b-filter-type, supp5a-behaviour-over-trials | Blocked: data missing |
| speech-r-count | Reproduced (101 under the non-event z-score; 377 electrodes against 468) |
| fig3b-hfb-increase-after | Reproduced |
| supp3a-pink-noise-no-hfb-change | Reproduced |
| supp5b-hfb-stable-over-session | Not reproduced |
| fig4a-coherence-after-higher | Reproduced (assumption) |
| fig4b-coherence-sentence-specific | Reproduced (assumption) |
| strf-r-count | Not reproduced (9 against 53) |
| included-electrode-count | Not reproduced (9 against 41) |
| fig8-mtf-matches-filter | Partially reproduced (assumption) |
| fig9-rms-increase | Partially reproduced |
| fig9-middle-generalizes-after | Reproduced |
| fig10-partial-correlation | Partially reproduced |
| fig10b-similarity-correlates | Not reproduced |
| fig10c-stg-location | Reproduced |
| supp3b-pink-noise-no-pcorr-change | Blocked: no included electrode in the pink noise subjects |
| supp6a-connectivity-increase | Partially reproduced (assumption) |
| supp6a-after-closer-to-middle | Reproduced (assumption) |
| supp6b-no-pac-change | Reproduced |
| supp7-envelope-coherence | Not reproduced |
| supp4-hfb-power-increase | Reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the HFB band edges, the filter orders, the
auditory spectrogram's compression and sharpening, the multitaper bandwidths and the wavelet
range are standard choices the paper does not fix; the z-score that defines Speech-R is the
code's z-score by non-event times, the only one of three readings that gives counts near the
paper's; the ridge parameter is the mode of the per-split best values below the top of the grid,
since the mode over all values falls at the top of any grid. The eSTRFs predict held-out trials
in far fewer electrodes than the paper reports (9 against 53), so every eSTRF claim rests on 9
electrodes from three subjects, and the pink noise eSTRF claim cannot be tested.

What was found about the data set: it holds 7 subjects with 416 channels (377 marked good)
against the paper's 524 recorded and 468 kept, with no mapping between its subject names and
the paper's S1 to S7; the stored values are not in the stated millivolts and their scale
differs by six orders of magnitude between S01 and the other subjects; the behavioural control
data are not included; the stimulus audio is included although the data set's own notebook says
it is not; events marked `bad` are not described.